# Run text2sql-eval from notebook

This notebook:
1. Detects the repository root reliably
2. Loads `config/config.yaml`
3. Builds a provider/model map from current config
4. Runs `run_experiment(...)` for available provider/model pairs


In [1]:
from pathlib import Path
import os

from text2sql_eval import run_experiment
from text2sql_eval.config import load_config

def detect_repo_root(start: Path) -> Path:
    env_root = os.getenv("TEXT2SQL_EVAL_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / "pyproject.toml").exists() and (candidate / "config/config.yaml").exists():
            return candidate

    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "config/config.yaml").exists():
            return candidate

    # Common local case: notebook kernel started from parent folder (e.g. /home/user/dev)
    sibling = start / "text2sql-eval"
    if (sibling / "pyproject.toml").exists() and (sibling / "config/config.yaml").exists():
        return sibling

    raise RuntimeError(
        f"Could not detect repo root from {start}. "
        "Set TEXT2SQL_EVAL_ROOT=/absolute/path/to/text2sql-eval"
    )

REPO_ROOT = detect_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)

CONFIG_PATH = REPO_ROOT / "config/config.yaml"

print("repo:", REPO_ROOT)
print("config exists:", CONFIG_PATH.exists())
print("dev exists:", (REPO_ROOT / "data/dev.json").exists())
print("db exists:", (REPO_ROOT / "data/database.sqlite").exists())


repo: /home/mileto/dev/text2sql-eval
config exists: True
dev exists: True
db exists: True


In [2]:
cfg = load_config(str(CONFIG_PATH))

provider_models: dict[str, list[str]] = {}
for model_cfg in cfg.models:
    provider_models.setdefault(model_cfg.provider, []).append(model_cfg.model)

print("Configured provider/model pairs:")
for provider, models in provider_models.items():
    print(f"- {provider}: {models}")


Configured provider/model pairs:
- fake: ['local-test']
- openai: ['gpt-4o']
- anthropic: ['claude-3-5-sonnet-20241022']


In [4]:
def provider_is_available(provider: str) -> bool:
    if provider == "fake":
        return False #### IMPORTANT ####: CHANGE TO FALSE TO NOT RUN THE 'FAKE' PROVIDER
    if provider == "openai":
        return bool(os.getenv("OPENAI_API_KEY"))
    if provider == "anthropic":
        return bool(os.getenv("ANTHROPIC_API_KEY"))
    return False

TRACK = "a"
LIMIT = 3

run_ids: dict[str, str] = {}
skipped: list[str] = []

for provider, models in provider_models.items():
    for model in models:
        pair = f"{provider}/{model}"
        if not provider_is_available(provider):
            skipped.append(pair)
            continue

        run_id = run_experiment(
            config_path=str(CONFIG_PATH),
            track=TRACK,
            limit=LIMIT,
            provider=provider,
            model=model,
        )
        run_ids[pair] = run_id

print("Completed runs:")
for pair, run_id in run_ids.items():
    artifact = REPO_ROOT / cfg.run_defaults.output_dir / run_id / "run.json"
    print(f"- {pair}: {artifact}")

if skipped:
    print("\nSkipped (missing credentials):")
    for pair in skipped:
        print(f"- {pair}")


Completed runs:

Skipped (missing credentials):
- fake/local-test
- openai/gpt-4o
- anthropic/claude-3-5-sonnet-20241022
